# 02 — Supervised training + offline pseudo-labeling (self-training)

Alternative approach to using the 2,079 unlabeled cores:

1. Train SegFormer-B4 on the 340 labeled cores (+ a flipped copy of each) for 15 epochs.
2. Predict the unlabeled set and keep only pixels with confidence ≥ 0.7 (the rest become `ignore_index = 0`).
3. Fine-tune on the pseudo-labeled set for 5 epochs, then predict the test set.

Written for Kaggle (`/kaggle/input/...` paths). The cells under *Helper Code* are scratch utilities for resuming from checkpoints.

See the [project README](../README.md) for the full write-up.

### Imports

In [ ]:
import os
import numpy as np
import random
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms

# Import SegFormer from Hugging Face Transformers
from transformers import SegformerForSemanticSegmentation

In [ ]:
class_dict = {
    1: (0, 77, 38),
    2: (255, 255, 255),
    3: (203, 20, 20),
    4: (191, 191, 147),
    5: (133, 223, 246),
    6: (159, 157, 12),
    7: (230, 193, 156),
    8: (139, 87, 42),
    9: (200, 200, 200),
}

In [ ]:
def create_label_mask(image, class_dict):
    image_array = np.asarray(image)
    reverse_class_dict = {tuple(v): k for k, v in class_dict.items()}
    # Reshape image to 2D array of RGB tuples
    rgb_tuples = image_array.reshape(-1, 3)
    # Initialize labels array
    labels = np.zeros(rgb_tuples.shape[0], dtype=np.uint8)
    # Vectorized lookup
    for rgb, class_label in reverse_class_dict.items():
        mask = np.all(rgb_tuples == rgb, axis=1)
        labels[mask] = class_label
    return labels.reshape(image_array.shape[:2])

### 1. Define Loss Functions: Dice Loss + Combined Loss

In [ ]:
def dice_loss(pred, target, eps=1e-6):
    """
    Computes the Dice loss.
    Args:
        pred: Tensor of shape (B, C, H, W) containing probabilities (after softmax).
        target: Tensor of shape (B, H, W) with ground-truth class indices.
    Returns:
        Dice loss (1 - Dice coefficient averaged over classes).
    """
    num_classes = pred.shape[1]
    # Convert target to one-hot encoding: shape (B, H, W, C) then permute to (B, C, H, W)
    target_one_hot = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    # Sum over batch and spatial dimensions
    dims = (0, 2, 3)
    intersection = torch.sum(pred * target_one_hot, dims)
    cardinality = torch.sum(pred + target_one_hot, dims)
    dice_score = (2. * intersection + eps) / (cardinality + eps)

    # Dice loss per class is (1 - dice_score); average over classes
    return 1 - dice_score.mean()

class CombinedLoss(nn.Module):
    """
    Combined loss: Cross-Entropy Loss + Dice Loss.
    """
    def __init__(self, weight_ce=1.0, weight_dice=1.0):
        super(CombinedLoss, self).__init__()
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.ce_loss = nn.CrossEntropyLoss(ignore_index=0)

    def forward(self, logits, target):
        # logits: (B, C, H, W)
        ce = self.ce_loss(logits, target)
        # Compute probabilities using softmax over class dimension
        probs = F.softmax(logits, dim=1)
        d_loss = dice_loss(probs, target)
        return self.weight_ce * ce + self.weight_dice * d_loss

### 2. Augmentation, Datasets, DataLoaders

#### UnlabeledDataset

In [ ]:
class UnlabeledDataset(Dataset):
    """
    Dataset for unlabeled images.
    Returns the image tensor and its filename.
    """
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.image_files = [f for f in os.listdir(data_dir) if f.endswith('.png')]

        self.inference_transform = transforms.Compose([
            transforms.Resize((1200, 300)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.658, 0.611, 0.486],
                                 std=[0.276, 0.287, 0.362])
                                 ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        image_path = os.path.join(self.data_dir, image_file)

        try:
            image = Image.open(image_path).convert("RGB")
            (orig_w, orig_h) = image.size
            image = self.inference_transform(image)
            return image, image_file, (orig_w, orig_h)

        except FileNotFoundError as e:
            print(f"Skipping {image_file}: {e}")
            return self.__getitem__((idx + 1) % len(self))  # Skip or retry

#### LabeledDataset

In [ ]:
class LabeledDataset(Dataset):
    """
    Dataset for labeled images with enhanced augmentation.
    """
    def __init__(self, image_dir, mask_dir, pseudos=False):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.pseudos = pseudos
        
        # List image files that are not masks
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('_img.png')]
        
        # Base transforms that are always applied
        self.image_transform = transforms.Compose([
            transforms.Resize((1200, 300)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.658, 0.611, 0.486],
                              std=[0.276, 0.287, 0.362])
        ])
        
        self.mask_transform = transforms.Compose([
            transforms.Resize((300, 75), interpolation=Image.NEAREST)
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        image_path = os.path.join(self.image_dir, image_file)

        mask_file = image_file.replace('_img.png', '_lab.png')
        mask_path = os.path.join(self.mask_dir, mask_file)

        try:
            # Load images
            image = Image.open(image_path).convert("RGB")
            mask = Image.open(mask_path)
            
            # Apply base transforms
            image = self.image_transform(image)
            mask = self.mask_transform(mask)
            if self.pseudos:
                label_mask = np.array(mask, dtype=np.int8).squeeze()
            else:
                label_mask = create_label_mask(mask, class_dict).astype(np.int32)

            return image, label_mask

        except FileNotFoundError as e:
            print(f"Skipping {image_file}: {e}")
            return self.__getitem__((idx + 1) % len(self))  # Skip or retry

#### AugmentedLabeledDataset

In [ ]:
class CustomAugmentation:
    def __init__(self, prob=0.5):
        self.prob = prob
        
    def __call__(self, image, mask):
        # Convert to PIL if they're tensors
        if isinstance(image, torch.Tensor):
            image = transforms.ToPILImage()(image)
        if isinstance(mask, torch.Tensor):
            mask = transforms.ToPILImage()(mask)
            
        # Random horizontal flip
        if random.random() < self.prob:
            image = transforms.functional.hflip(image)
            mask = transforms.functional.hflip(mask)

        # Random vertical flip
        else:
            image = transforms.functional.vflip(image)
            mask = transforms.functional.vflip(mask)
            
        return image, mask

class AugmentedLabeledDataset(Dataset):
    """
    Dataset for labeled images with enhanced augmentation.
    """
    def __init__(self, data_dir):
        self.data_dir = data_dir
        
        # List image files that are not masks
        self.image_files = [f for f in os.listdir(data_dir) if f.endswith('_img.png')]
        
        # Define augmentations
        self.custom_aug = CustomAugmentation(prob=0.7)
        
        # Base transforms that are always applied
        self.image_transform = transforms.Compose([
            transforms.Resize((1200, 300)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.658, 0.611, 0.486],
                              std=[0.276, 0.287, 0.362])
        ])
        
        self.mask_transform = transforms.Compose([
            transforms.Resize((300, 75), interpolation=Image.NEAREST)
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        image_path = os.path.join(self.data_dir, image_file)
        mask_file = image_file.replace('_img.png', '_lab.png')
        mask_path = os.path.join(self.data_dir, mask_file)

        try:
            # Load images
            image = Image.open(image_path).convert("RGB")
            mask = Image.open(mask_path)
            
            # Apply augmentations if enabled
            image, mask = self.custom_aug(image, mask)
            
            # Apply base transforms
            image = self.image_transform(image)
            mask = self.mask_transform(mask)
            label_mask = create_label_mask(mask, class_dict).astype(np.int32)

            return image, label_mask

        except FileNotFoundError as e:
            print(f"Skipping {image_file}: {e}")
            return self.__getitem__((idx + 1) % len(self))  # Skip or retry

#### Dataloaders and paths

In [ ]:
labeled_data_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'train', 'train')
unlabeled_data_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'train_unlabeled')
test_data_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'test', 'test')
pseudo_image_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'train_unlabeled')
pseudo_mask_dir = os.path.join('/kaggle', 'working', 'pseudo_labels')

### 3. Model Training, Testing Finetuning

#### Model Initialize

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set the number of classes (adjust according to your dataset)
NUM_CLASSES = 10

# Load a pretrained SegFormer model and adjust the number of labels
model_name = "nvidia/segformer-b4-finetuned-ade-512-512"
model = SegformerForSemanticSegmentation.from_pretrained(model_name, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)
model.to(device)

# Instantiate the combined loss and the optimizer
criterion = CombinedLoss(weight_ce=1.0, weight_dice=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

#### Model Training

In [ ]:
def train_model(labeled_loader, epochs, train_iter):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, masks in tqdm(labeled_loader):
    
            images = images.to(device)       # Shape: (B, 3, H, W)
            masks = masks.long()
            masks = masks.to(device)         # Shape: (B, H, W)
    
            optimizer.zero_grad()
            # For Hugging Face SegFormer, pass the images as "pixel_values"
            outputs = model(pixel_values=images).logits  # Shape: (B, NUM_CLASSES, H, W)
    
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
    
        avg_loss = running_loss / len(labeled_loader)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")
    
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict()
    }
    torch.save(checkpoint, f"labeled_training_iter_{train_iter}.pth")
    print(f"Model Trained and saved as labeled_training_iter_{train_iter}.pth")

#### Inference on Test Data

In [ ]:
def test_model(test_loader, train_iter, train_step):
    model.eval()
    image_names = []
    predicted_masks = []
    output_file = f"test_predictions_{train_step}_{train_iter}.npz"

    with torch.no_grad():
        for image, image_file, image_size in tqdm(test_loader):
            image = image.to(device)
            logits = model(pixel_values=image).logits  # (1, NUM_CLASSES, H, W)
            # Converting back to the original shape
            logits_upsampled = F.interpolate(
                logits, size=(image_size[1], image_size[0]), mode="bilinear", align_corners=False)
            # Convert logits to probabilities
            probs = F.softmax(logits_upsampled, dim=1)  # (1, NUM_CLASSES, H, W)

            # For each pixel, get the predicted class (argmax)
            pseudo_mask = torch.argmax(probs, dim=1)
            pseudo_mask = pseudo_mask.squeeze().cpu().numpy().astype(np.uint8)

            # Store using the image filename as key
            predicted_masks.append(pseudo_mask)
            image_names.append(image_file[0].replace("_img.png", "_lab"))

    with open(output_file, 'wb') as f:  # Use 'wb' for binary write mode
        np.savez_compressed(f, **dict(zip(image_names, predicted_masks))) #Use np.savez for uncompressed

    print(f"Inference complete! Masks saved to {output_file}")

#### Generate Pseudo Masks with Confidences

In [ ]:
def pseudo_label_generation(unlabeled_loader, train_iter, confidence_threshold):
    model.eval()
    pseudo_masks = {}  # Dictionary to store results for each image
    ignore_index = 0  # Special value for unconfident pixels
    total_pixels = 0
    confident_pixels = 0
    
    with torch.no_grad():
        for image, image_file, image_size in tqdm(unlabeled_loader):
            image = image.to(device)
            output = model(pixel_values=image).logits  # (1, NUM_CLASSES, H, W)
            # Convert logits to probabilities
            probs = F.softmax(output, dim=1)  # (1, NUM_CLASSES, H, W)
    
            # Converting back to the original shape
            probs = F.interpolate(probs, size=(image_size[1], image_size[0]), mode="bilinear", align_corners=False)
    
            # For each pixel, get the predicted class (argmax) and the corresponding confidence (max probability)
            confidence, pseudo_mask = torch.max(probs, dim=1)  # Both tensors of shape (1, H, W)

            # Apply confidence threshold
            confidence_mask = confidence >= confidence_threshold
            pseudo_mask[~confidence_mask] = ignore_index  # Set unconfident pixels to 0

            # Update statistics
            confident_pixels += confidence_mask.sum().item()
            total_pixels += confidence_mask.numel()

            pseudo_mask = pseudo_mask.squeeze(0).cpu().numpy().astype(np.uint8)
    
            # Store using the image filename as key
            pseudo_masks[image_file[0]] = {'mask': pseudo_mask}
    
    # Calculate confidence metrics
    confidence_ratio = confident_pixels / total_pixels if total_pixels > 0 else 0
    print(f"Confidence thresholding kept {confidence_ratio:.2%} of pixels")

    # Optionally, save the pseudo masks and confidence maps to disk.
    output_dir = os.path.join('pseudo_labels')
    os.makedirs(output_dir, exist_ok=True)
    
    for filename, data_dict in pseudo_masks.items():
        mask = data_dict['mask']
        mask_img = Image.fromarray(mask)
        mask_filename = filename.replace('_img.png', '_lab.png')
        mask_img.save(os.path.join(output_dir, mask_filename))
    
    print(f"Pseudo labeling {train_iter} complete. Pseudo masks and confidence maps saved in:", output_dir)

#### Fine Tune on Pseudo Labeled Data

In [ ]:
def finetune_model(pseudo_loader, epochs, train_iter):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, pseudo_masks in tqdm(pseudo_loader):
            images = images.to(device)           # shape: (B, 3, H, W)
            pseudo_masks = pseudo_masks.to(device) # shape: (B, H, W)
    
            optimizer.zero_grad()
    
            # Forward pass through the model
            outputs = model(pixel_values=images).logits  # Shape: (B, NUM_CLASSES, h_out, w_out)
    
            # Resize the pseudo masks to match the output dimensions
            pseudo_masks_resized = F.interpolate(
                pseudo_masks.unsqueeze(1).float(), size=outputs.shape[2:], mode="nearest"
            ).squeeze(1).long()
    
            # Compute the loss on pseudo-labeled data
            loss = criterion(outputs, pseudo_masks_resized)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
    
        avg_loss = running_loss / len(pseudo_loader)
        print(f"Fine-tuning Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict()
        }
    torch.save(checkpoint, f"unlabeled_training_iter_{train_iter}.pth")
    print(f"Model Finetuned and saved as unlabeled_training_iter_{train_iter}.pth")

#### All Together

In [ ]:
for i in range(1):
    # model training
    aug_labeled_dataset = AugmentedLabeledDataset(labeled_data_dir)
    labeled_dataset = LabeledDataset(image_dir=labeled_data_dir, mask_dir=labeled_data_dir)
    labeled_dataset = ConcatDataset([labeled_dataset, aug_labeled_dataset])
    labeled_loader = DataLoader(labeled_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
    train_model(labeled_loader, epochs=15, train_iter=i)

    # model testing
    test_dataset = UnlabeledDataset(test_data_dir)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    test_model(test_loader, train_iter=1, train_step='model_trained')

    # pseudo label generation
    unlabeled_dataset = UnlabeledDataset(unlabeled_data_dir)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=1, shuffle=False, num_workers=2)
    pseudo_label_generation(unlabeled_loader, train_iter=i, confidence_threshold=0.7)

    # model finetuning on pseudo labels
    pseudo_dataset = LabeledDataset(image_dir=pseudo_image_dir, mask_dir=pseudo_mask_dir, pseudos=True)
    pseudo_loader = DataLoader(pseudo_dataset, batch_size=4, shuffle=True, num_workers=2)
    finetune_model(pseudo_loader, epochs=5, train_iter=i)

    # model testing
    test_dataset = UnlabeledDataset(test_data_dir)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    test_model(test_loader, train_iter=1, train_step='model_finetuned')

### Helper Code

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set the number of classes (adjust according to your dataset)
NUM_CLASSES = 9

checkpoint = torch.load("/kaggle/input/segformer-finetuned/pytorch/default/1/labeled_training_2.pth", weights_only=False)
# Load a pretrained SegFormer model and adjust the number of labels
model_name = "nvidia/segformer-b4-finetuned-ade-512-512"
model = SegformerForSemanticSegmentation.from_pretrained(model_name, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)
model.load_state_dict(checkpoint["model_state"])
model.to(device)

# Instantiate the combined loss and the optimizer
criterion = CombinedLoss(weight_ce=1.0, weight_dice=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state"])

start_epoch = checkpoint["epoch"] + 1  # Resume from the next epoch

In [ ]:
def load_arrays_from_npz(filename):
    data = np.load(filename)
    return dict(data)  # Convert the NpzFile object to a dictionary

# Example loading
loaded_arrays = load_arrays_from_npz("test_predictions.npz")

In [ ]:
arr1 = loaded_arrays['1_lab']

In [ ]:
checkpoint = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict()
}
torch.save(checkpoint, "labeled_training_11.pth")

In [ ]:
checkpoint = torch.load("checkpoint.pth")
model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"] + 1  # Resume from the next epoch